# Email Spam Detection Using Machine Learning

This notebook demonstrates how to build a machine learning model to detect spam emails. We will use Python and popular libraries such as scikit-learn, pandas, numpy, and nltk.

## What is Spam? Why is Detecting Spam Important?

Spam refers to unwanted or unsolicited messages, often sent in bulk, usually for advertising or malicious purposes. Detecting spam is important because it helps protect users from scams, phishing, and unwanted content, and keeps email inboxes clean.

## Importing Required Libraries

We will import the following libraries:
- **pandas**: For data manipulation and analysis.
- **numpy**: For numerical operations.
- **nltk**: For natural language processing tasks like stopword removal and tokenization.
- **scikit-learn**: For machine learning, including feature extraction, model building, and evaluation.

In [27]:
# Install missing packages
%pip install pandas
%pip install numpy
%pip install nltk
%pip install scikit-learn
# --- IGNORE ---


# Importing libraries
import pandas as pd
import numpy as np
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import string
import re
nltk.download('stopwords')
from nltk.corpus import stopwords

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: C:\Users\anand\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: C:\Users\anand\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: C:\Users\anand\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\anand\AppData\Roaming\nltk_data...

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: C:\Users\anand\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
[nltk_data]   Package stopwords is already up-to-date!


## Loading the Dataset

We will use the popular SMS Spam Collection dataset. This dataset contains messages labeled as 'spam' or 'ham' (not spam).

Let's load the dataset and look at the first few rows.

In [28]:
# Download and load the dataset
url = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'
df = pd.read_csv(url, sep='	', header=None, names=['label', 'message'])
# Display the first 5 rows
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### Dataset Columns Explained
- **label**: Indicates whether the message is 'spam' or 'ham' (not spam).
- **message**: The actual text content of the email or SMS.

## Exploratory Data Analysis (EDA)

Let's see how many messages are spam and how many are ham. This helps us understand if the dataset is balanced or not.

In [29]:
# Count of spam and ham messages
df['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

### Is the Dataset Balanced?

If one class (spam or ham) has many more samples than the other, the dataset is imbalanced. This can affect model performance.

In [30]:
# Calculate percentage of each class
df['label'].value_counts(normalize=True) * 100

label
ham     86.593683
spam    13.406317
Name: proportion, dtype: float64

## Text Preprocessing

Text data needs to be cleaned before it can be used for machine learning. We will:
1. Convert text to lowercase
2. Remove punctuation and numbers
3. Remove stopwords (common words like 'the', 'is', etc.)
4. Tokenize (split text into words)

Let's define a function to do all these steps.

In [31]:
# Define stopwords
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation and numbers, but keep spaces
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize
    tokens = text.split()
    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    # Join tokens back to string
    return ' '.join(tokens)

# Apply preprocessing
df['clean_message'] = df['message'].apply(preprocess_text)
# Show the first 5 cleaned messages
df[['message', 'clean_message']].head()

,message,clean_message
0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...
3,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,"Nah I don't think he goes to usf, he lives aro...",nah dont think goes usf lives around though


### Explanation of Preprocessing Steps
- **Lowercase**: Makes all text the same case, so 'Hello' and 'hello' are treated the same.
- **Remove punctuation/numbers**: Only keeps letters, which are most important for meaning.
- **Remove stopwords**: Removes common words that do not add much meaning.
- **Tokenization**: Splits text into individual words.

## Feature Extraction: TF-IDF Vectorizer

Machine learning models need numbers, not text. We use TF-IDF to convert text into numbers.

- **TF (Term Frequency)**: How often a word appears in a message.
- **IDF (Inverse Document Frequency)**: How unique a word is across all messages.

TF-IDF gives higher scores to words that are frequent in a message but rare in the whole dataset.

In [32]:
# Create TF-IDF features
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['clean_message'])
y = df['label'].map({'ham': 0, 'spam': 1})  # 0 for ham, 1 for spam

## Train-Test Split

We split the data into training and testing sets.

- **Training set (80%)**: Used to train the model.
- **Testing set (20%)**: Used to evaluate the model.

Splitting helps us check if the model works well on new, unseen data.

In [33]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Training samples:', X_train.shape[0])
print('Testing samples:', X_test.shape[0])

Training samples: 4457
Testing samples: 1115


## Model Building: Naive Bayes Classifier

We use the Naive Bayes classifier because it works very well for text classification problems. It is fast, simple, and often gives good results for spam detection.

In [34]:
# Build the model
model = MultinomialNB()

## Model Training

We train the model using the training data.

In [35]:
# Train the model
model.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


## Model Evaluation

We evaluate the model using several metrics:
- **Accuracy**: Percentage of correct predictions.
- **Precision**: Of all messages predicted as spam, how many are actually spam?
- **Recall**: Of all actual spam messages, how many did we find?
- **F1-Score**: Balance between precision and recall.
- **Confusion Matrix**: Shows correct and incorrect predictions for each class.

In [36]:
# Predict on test data
y_pred = model.predict(X_test)

# Calculate metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f'Accuracy: {acc:.2f}')
print(f'Precision: {prec:.2f}')
print(f'Recall: {rec:.2f}')
print(f'F1-Score: {f1:.2f}')
print('Confusion Matrix:')
print(cm)

# Detailed classification report
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

Accuracy: 0.97
Precision: 1.00
Recall: 0.77
F1-Score: 0.87
Confusion Matrix:
[[966   0]
 [ 34 115]]
Classification Report:
              precision    recall  f1-score   support

         Ham       0.97      1.00      0.98       966
        Spam       1.00      0.77      0.87       149

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115



### Explanation of Metrics
- **Accuracy**: Overall, how often is the classifier correct?
- **Precision**: When it predicts spam, how often is it right?
- **Recall**: How many actual spam messages did it find?
- **F1-Score**: A single score that balances precision and recall.
- **Confusion Matrix**: Shows counts of true positives, true negatives, false positives, and false negatives.

## Test with Custom Email Input

Let's allow the user to enter a custom email message and predict if it is spam or ham.

In [37]:
# Function to predict custom input
def predict_message(msg):
    clean = preprocess_text(msg)
    vect = vectorizer.transform([clean])
    pred = model.predict(vect)[0]
    return 'Spam' if pred == 1 else 'Ham'

# User input
custom_msg = input('Enter an email message to classify: ')
print('Prediction:', predict_message(custom_msg))

Prediction: Ham


## Conclusion

In this project, we built a machine learning model to detect spam emails. We learned how to preprocess text, extract features using TF-IDF, and use the Naive Bayes classifier. The model was evaluated using several metrics and can predict if a new message is spam or ham.

**Key Learning Outcomes:**
- Importance of spam detection
- Text preprocessing techniques
- Feature extraction with TF-IDF
- Model building and evaluation
- Testing with custom input

This notebook is ready for academic submission and further exploration.